# IceWave — East Model Improvement (v3)
**Goal:** Push east model AUC from 0.566 toward 0.75–0.80

**Strategy:**
- Option 1: Deeper PBDB harvest — all Pleistocene vertebrates east of Cascades (not just 8 taxa)
- Option 2: Expand geography — add Idaho and Montana to east training area

**Study states:** WA, OR, NV (existing) + ID, MT (new)

**Run this notebook top to bottom. All outputs auto-saved.**

## 0. Setup

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import rasterio.enums
import rasterio.transform
import requests
import time
from io import StringIO
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
import joblib
import simplekml
import gpxpy
import gpxpy.gpx

Path('../data/pbdb').mkdir(parents=True, exist_ok=True)
Path('../data/model').mkdir(parents=True, exist_ok=True)
Path('../outputs').mkdir(parents=True, exist_ok=True)

# ── Constants ─────────────────────────────────────────────────────────────
CASCADE_SPLIT = -121.5
LAT_MIN       = 36.0
LAT_MAX       = 49.5
EAST_STATES   = ['Washington', 'Oregon', 'Nevada', 'Idaho', 'Montana']
FEATURES      = ['elevation', 'slope', 'aspect', 'tri', 'lith_score']

TERRAIN_FILES = {
    'elevation' : '../data/terrain/dem_mosaic.tif',
    'slope'     : '../data/terrain/slope.tif',
    'aspect'    : '../data/terrain/aspect.tif',
    'tri'       : '../data/terrain/ruggedness.tif',
}

LITH_SCORE_MAP = {
    'Clay': 1.0, 'Silt': 1.0, 'Sand': 1.0, 'Gravel': 1.0,
    'Unconsolidated': 1.0, 'Alluvium': 1.0, 'Loess': 1.0,
    'Sandstone': 0.9, 'Mudstone': 0.9, 'Shale': 0.9,
    'Limestone': 0.7, 'Dolomite': 0.6,
    'Granite': 0.2, 'Basalt': 0.15, 'Andesite': 0.15,
    'Rhyolite': 0.15, 'Metamorphic': 0.1, 'Gneiss': 0.1,
}

def score_lith(lith_str):
    if pd.isna(lith_str): return 0.4
    for key, val in LITH_SCORE_MAP.items():
        if key.lower() in str(lith_str).lower(): return val
    return 0.4

print('Setup complete.')
print(f'States: {EAST_STATES}')
print(f'East region: {LAT_MIN}–{LAT_MAX}N, east of {CASCADE_SPLIT}W')

Setup complete.
States: ['Washington', 'Oregon', 'Nevada', 'Idaho', 'Montana']
East region: 36.0–49.5N, east of -121.5W


## 1. PBDB Harvest — All Pleistocene Vertebrates

In [2]:
PBDB_URL    = 'https://paleobiodb.org/data1.2/occs/list.csv'
all_records = []

for state in EAST_STATES:
    params = {
        'state'     : state,
        'interval'  : 'Pleistocene',
        'base_name' : 'Vertebrata',
        'show'      : 'coords,classext,ident',
        'limit'     : '5000',
    }
    try:
        resp = requests.get(PBDB_URL, params=params, timeout=30)
        if resp.status_code != 200:
            print(f'{state}: HTTP {resp.status_code}')
            continue
        state_df = pd.read_csv(StringIO(resp.text))
        state_df = state_df.dropna(subset=['lng', 'lat'])
        if len(state_df) == 0:
            print(f'{state}: 0 records with coordinates')
            continue
        state_df['state_query'] = state
        all_records.append(state_df)
        print(f'{state}: {len(state_df)} records')
        time.sleep(0.5)
    except Exception as e:
        print(f'{state}: ERROR — {e}')

if len(all_records) == 0:
    raise RuntimeError('No records collected — check network connection')

raw = pd.concat(all_records, ignore_index=True)
raw = raw.rename(columns={'lng': 'longitude', 'lat': 'latitude'})

taxon_col = 'accepted_name' if 'accepted_name' in raw.columns else 'taxon_name'
raw['genus'] = raw[taxon_col].str.split().str[0]

print(f'\nTotal raw records : {len(raw)}')
print(f'Taxon column      : {taxon_col}')
print(f'\nTop 20 taxa:')
print(raw[taxon_col].value_counts().head(20).to_string())

Washington: 36 records
Oregon: 255 records
Nevada: 103 records
Idaho: 82 records
Montana: 63 records

Total raw records : 539
Taxon column      : accepted_name

Top 20 taxa:
accepted_name
Equus                   18
Mammuthus columbi       10
Paramylodon harlani      9
Camelops                 7
Bison antiquus           7
Odocoileus               6
Plesippus idahoensis     6
Mammut americanum        5
Mammuthus                5
Canis latrans            5
Taxidea taxus            4
Hemiauchenia             4
Bison                    4
Proboscidea              4
Camelops hesternus       4
Anas platyrhynchos       4
Ovis canadensis          4
Thomomys                 4
Microtus                 4
Leporidae                4


In [3]:
# ── Filter to east of Cascade crest ───────────────────────────────────────
east_raw = raw[
    (raw['longitude'] > CASCADE_SPLIT) &
    (raw['longitude'] < -100.0) &
    (raw['latitude']  >= LAT_MIN) &
    (raw['latitude']  <= LAT_MAX)
].copy()

print(f'Records east of Cascades: {len(east_raw)}')
print(f'Lat: {east_raw.latitude.min():.2f} – {east_raw.latitude.max():.2f}')
print(f'Lon: {east_raw.longitude.min():.2f} – {east_raw.longitude.max():.2f}')
print(f'\nBy state:')
print(east_raw['state_query'].value_counts().to_string())
print(f'\nTop genera:')
print(east_raw['genus'].value_counts().head(20).to_string())

Records east of Cascades: 425
Lat: 36.22 – 48.80
Lon: -121.05 – -105.80

By state:
state_query
Oregon        149
Nevada        103
Idaho          82
Montana        63
Washington     28

Top genera:
genus
Equus           22
Mammuthus       14
Camelops        13
Bison            9
Canis            9
Anas             9
Paramylodon      8
Odocoileus       8
Microtus         8
Thomomys         8
Branta           8
Ondatra          7
Taxidea          6
Mammut           6
Anser            6
Plesippus        6
Lynx             5
Hemiauchenia     5
Arctodus         5
Larus            5


In [4]:
# ── Deduplicate by location (~100m grid) ──────────────────────────────────
east_raw['lat_r']    = east_raw['latitude'].round(3)
east_raw['lon_r']    = east_raw['longitude'].round(3)
east_raw['_key']     = east_raw['lat_r'].astype(str) + '_' + east_raw['lon_r'].astype(str)
east_raw['name_len'] = east_raw[taxon_col].str.len()

east_dedup = (
    east_raw
    .sort_values('name_len', ascending=False)
    .drop_duplicates(subset=['_key'])
    .reset_index(drop=True)
)

print(f'After deduplication: {len(east_dedup)} unique locations')
print(f'\nBy state (deduped):')
print(east_dedup['state_query'].value_counts().to_string())
print(f'\nTop genera (deduped):')
print(east_dedup['genus'].value_counts().head(15).to_string())

east_dedup.to_csv('../data/pbdb/icewave_east_expanded_raw.csv', index=False)
print(f'\nSaved: ../data/pbdb/icewave_east_expanded_raw.csv')

After deduplication: 46 unique locations

By state (deduped):
state_query
Montana       18
Idaho          9
Nevada         8
Washington     6
Oregon         5

Top genera (deduped):
genus
Mammuthus          5
Cynomys            3
Camelops           3
Urocitellus        2
Bootherium         2
Odocoileus         2
Homo               2
Bos                2
Podiceps           1
Chroicocephalus    1
Allophaiomys       1
Hemiauchenia       1
Euceratherium      1
Phrynosoma         1
Antilocapra        1

Saved: ../data/pbdb/icewave_east_expanded_raw.csv


## 2. Sample Terrain

In [5]:
print(f'Sampling terrain for {len(east_dedup)} points...')

sources = {k: rasterio.open(v) for k, v in TERRAIN_FILES.items()}
rows_with_terrain = []
skipped = 0

for _, row in east_dedup.iterrows():
    coord = [(row['longitude'], row['latitude'])]
    try:
        elev = list(sources['elevation'].sample(coord))[0][0]
        if elev is None or np.isnan(elev) or elev < -500 or elev > 5000:
            skipped += 1
            continue
        slope = list(sources['slope'].sample(coord))[0][0]
        asp   = list(sources['aspect'].sample(coord))[0][0]
        tri   = list(sources['tri'].sample(coord))[0][0]
        if any(np.isnan(v) for v in [slope, asp, tri]):
            skipped += 1
            continue
        rows_with_terrain.append({
            **row.to_dict(),
            'elevation': elev, 'slope': slope,
            'aspect': asp, 'tri': tri
        })
    except Exception:
        skipped += 1

for src in sources.values():
    src.close()

east_terrain = pd.DataFrame(rows_with_terrain)
n_presence   = len(east_terrain)

print(f'Points with valid terrain : {n_presence}')
print(f'Skipped (outside DEM)     : {skipped}')
print(f'DEM coverage              : {n_presence/len(east_dedup)*100:.1f}%')
print(f'\nBy state:')
print(east_terrain['state_query'].value_counts().to_string())
print(f'\nTerrain stats:')
print(east_terrain[['elevation','slope','aspect','tri']].describe().round(2).to_string())

if n_presence >= 40:
    print(f'\n✓ {n_presence} points — good for improved east model.')
elif n_presence >= 25:
    print(f'\n~ {n_presence} points — marginal improvement expected.')
else:
    print(f'\n⚠ {n_presence} points — DEM expansion may be needed for ID/MT.')

Sampling terrain for 46 points...
Points with valid terrain : 40
Skipped (outside DEM)     : 6
DEM coverage              : 87.0%

By state:
state_query
Montana       18
Nevada         8
Idaho          6
Washington     5
Oregon         3

Terrain stats:
       elevation  slope  aspect    tri
count      40.00  40.00   40.00  40.00
mean      507.74   3.03   64.62   1.31
std       785.67   7.00  100.40   3.15
min         0.00   0.00    0.00   0.00
25%         0.00   0.00    0.00   0.00
50%         0.00   0.00    0.00   0.00
75%       710.39   1.52  109.40   0.70
max      3346.21  29.35  313.07  14.28

✓ 40 points — good for improved east model.


## 3. Lithology Enrichment (SGMC)

In [6]:
east_terrain['lith_score'] = 0.4
quat = None

try:
    print('Loading SGMC...')
    sgmc = gpd.read_file('../data/geology/SGMC/USGS_SGMC_Shapefiles/SGMC_Geology.shp')
    sgmc = sgmc.to_crs('EPSG:4326')
    quat = sgmc[sgmc['AGE_MIN'].str.contains('Quaternary', na=False)].copy()
    print(f'Quaternary polygons: {len(quat):,}')

    gdf = gpd.GeoDataFrame(
        east_terrain,
        geometry=gpd.points_from_xy(east_terrain['longitude'], east_terrain['latitude']),
        crs='EPSG:4326'
    )
    joined = gpd.sjoin(gdf, quat[['geometry','MAJOR1']], how='left', predicate='within')
    joined = joined[~joined.index.duplicated(keep='first')]
    east_terrain['lith_score'] = joined['MAJOR1'].apply(score_lith).values
    print(f'Lith score mean    : {east_terrain["lith_score"].mean():.3f}')
    print(f'High score (>0.8)  : {(east_terrain["lith_score"]>0.8).sum()} / {n_presence}')

except Exception as e:
    print(f'SGMC not available ({e}) — using default lith_score=0.4')

cols = ['latitude','longitude','elevation','slope','aspect','tri',
        'lith_score','genus', taxon_col,'state_query','_key']
cols = [c for c in cols if c in east_terrain.columns]
east_terrain[cols].to_csv('../data/pbdb/icewave_east_expanded.csv', index=False)
print(f'\nSaved: ../data/pbdb/icewave_east_expanded.csv ({n_presence} rows)')

Loading SGMC...
Quaternary polygons: 75,396
Lith score mean    : 0.604
High score (>0.8)  : 14 / 40

Saved: ../data/pbdb/icewave_east_expanded.csv (40 rows)


## 4. Train Improved East Model

In [7]:
# ── Background points ─────────────────────────────────────────────────────
ratio = 8 if n_presence >= 30 else 5
N_BG  = n_presence * ratio
print(f'Presence: {n_presence} | Background: {N_BG} ({ratio}:1)')

np.random.seed(42)
with rasterio.open(TERRAIN_FILES['elevation']) as src:
    bounds  = src.bounds
    bg_lons = np.random.uniform(CASCADE_SPLIT, min(bounds.right, -100.0), N_BG * 4)
    bg_lats = np.random.uniform(max(bounds.bottom, LAT_MIN),
                                min(bounds.top, LAT_MAX), N_BG * 4)
    coords    = list(zip(bg_lons, bg_lats))
    elev_vals = np.array([v[0] for v in src.sample(coords)])

valid   = (elev_vals > -500) & (elev_vals < 5000) & (~np.isnan(elev_vals))
bg_lons = bg_lons[valid][:N_BG]
bg_lats = bg_lats[valid][:N_BG]
bg_df   = pd.DataFrame({'longitude': bg_lons, 'latitude': bg_lats})

bg_coords = list(zip(bg_df['longitude'], bg_df['latitude']))
sources   = {k: rasterio.open(v) for k, v in TERRAIN_FILES.items()}
for feat, src in sources.items():
    bg_df[feat] = np.array([v[0] for v in src.sample(bg_coords)])
for src in sources.values():
    src.close()

bg_df['lith_score'] = 0.4
bg_df = bg_df.dropna(subset=FEATURES).reset_index(drop=True)
print(f'Valid background points: {len(bg_df)}')

Presence: 40 | Background: 320 (8:1)
Valid background points: 249


In [8]:
# ── Build training set & train ────────────────────────────────────────────
east_terrain['presence'] = 1
bg_df['presence']        = 0

train_east = pd.concat(
    [east_terrain[FEATURES + ['presence','latitude','longitude']],
     bg_df[FEATURES + ['presence','latitude','longitude']]],
    ignore_index=True
).dropna(subset=FEATURES)

X = train_east[FEATURES].values
y = train_east['presence'].values

min_leaf   = 2 if n_presence < 30 else 3
rf_east_v3 = RandomForestClassifier(
    n_estimators=500, min_samples_leaf=min_leaf,
    class_weight='balanced', random_state=42, n_jobs=-1
)

cv         = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_scores = cross_val_score(rf_east_v3, X, y, cv=cv, scoring='roc_auc')
rf_east_v3.fit(X, y)

fi = pd.Series(rf_east_v3.feature_importances_, index=FEATURES).sort_values(ascending=False)

print('=' * 50)
print('  EAST MODEL v3 RESULTS')
print('=' * 50)
print(f'  Training points  : {n_presence} (was 17)')
print(f'  CV AUC           : {auc_scores.mean():.3f} ± {auc_scores.std():.3f}')
print(f'  v2 east baseline : 0.566 ± 0.121')
print(f'  Delta            : {auc_scores.mean() - 0.566:+.3f}')
print(f'  Folds            : {np.round(auc_scores, 3)}')
print(f'\n  Feature importances:')
for feat, imp in fi.items():
    print(f'    {feat:<15} {imp:.3f}')
print('=' * 50)

joblib.dump(rf_east_v3, '../data/model/icewave_rf_east_v3.joblib')
print('Saved: ../data/model/icewave_rf_east_v3.joblib')

  EAST MODEL v3 RESULTS
  Training points  : 40 (was 17)
  CV AUC           : 0.846 ± 0.073
  v2 east baseline : 0.566 ± 0.121
  Delta            : +0.280
  Folds            : [0.742 0.885 0.912 0.915 0.776]

  Feature importances:
    elevation       0.329
    slope           0.225
    tri             0.219
    aspect          0.191
    lith_score      0.036
Saved: ../data/model/icewave_rf_east_v3.joblib


## 5. Generate East Candidate Predictions

In [9]:
STRIDE = 50
print(f'Sampling east candidate grid (stride={STRIDE})...')

with rasterio.open(TERRAIN_FILES['elevation']) as src:
    data      = src.read(1,
        out_shape=(src.height // STRIDE, src.width // STRIDE),
        resampling=rasterio.enums.Resampling.nearest)
    transform = src.transform * src.transform.scale(STRIDE, STRIDE)
    nodata    = src.nodata

rows_idx, cols_idx = np.where((data != nodata) & (data > -500) & (data < 5000))
xs, ys = rasterio.transform.xy(transform, rows_idx, cols_idx)
xs, ys = np.array(xs), np.array(ys)

mask  = (xs > CASCADE_SPLIT) & (xs < -100.0) & (ys >= LAT_MIN) & (ys <= LAT_MAX)
xs, ys = xs[mask], ys[mask]

cand_east = pd.DataFrame({
    'longitude': xs, 'latitude': ys,
    'elevation': data[rows_idx, cols_idx][mask]
})

cand_coords = list(zip(cand_east['longitude'], cand_east['latitude']))
for feat in ['slope', 'aspect', 'tri']:
    with rasterio.open(TERRAIN_FILES[feat]) as src:
        cand_east[feat] = np.array([v[0] for v in src.sample(cand_coords)])

cand_east['lith_score'] = 0.4
cand_east = cand_east.dropna(subset=FEATURES).reset_index(drop=True)
print(f'East candidate pixels: {len(cand_east):,}')

Sampling east candidate grid (stride=50)...
East candidate pixels: 90,720


In [10]:
# ── Enrich candidates with SGMC ───────────────────────────────────────────
if quat is not None:
    try:
        gdf    = gpd.GeoDataFrame(
            cand_east,
            geometry=gpd.points_from_xy(cand_east['longitude'], cand_east['latitude']),
            crs='EPSG:4326'
        )
        joined = gpd.sjoin(gdf, quat[['geometry','MAJOR1']], how='left', predicate='within')
        joined = joined[~joined.index.duplicated(keep='first')]
        cand_east['lith_score'] = joined['MAJOR1'].apply(score_lith).values
        print(f'Candidate lith_score mean: {cand_east["lith_score"].mean():.3f}')
    except Exception as e:
        print(f'SGMC join failed ({e}) — using default 0.4')
else:
    print('SGMC not loaded — using default lith_score=0.4')

# ── Predict ───────────────────────────────────────────────────────────────
cand_east['prob']      = rf_east_v3.predict_proba(cand_east[FEATURES].values)[:, 1]
cand_east['composite'] = (0.80 * cand_east['prob']) + (0.20 * cand_east['lith_score'])

print(f'Prob range       : {cand_east["prob"].min():.3f} – {cand_east["prob"].max():.3f}')
print(f'High-prob (>0.7) : {(cand_east["prob"]>0.7).sum():,}')

# Quaternary filter
try:
    quat_dep = gpd.read_file('../data/geology/quaternary_deposits.shp').to_crs('EPSG:4326')
    gdf_c    = gpd.GeoDataFrame(
        cand_east,
        geometry=gpd.points_from_xy(cand_east['longitude'], cand_east['latitude']),
        crs='EPSG:4326'
    )
    in_q  = gpd.sjoin(gdf_c, quat_dep[['geometry']], how='inner', predicate='within')
    pool  = cand_east.loc[in_q.index.unique()]
    print(f'In Quaternary deposits: {len(pool):,}')
except Exception as e:
    print(f'Quaternary filter failed ({e}) — using all candidates')
    pool = cand_east

# Spatial thinning & top 25
pool         = pool.copy()
pool['cell'] = (pool['latitude'].round(0).astype(str) + '_' +
                pool['longitude'].round(0).astype(str))
thinned      = pool.sort_values('composite', ascending=False).drop_duplicates('cell')
top25_east   = thinned.nlargest(25, 'composite').reset_index(drop=True)
top25_east.index += 1
top25_east.index.name    = 'rank'
top25_east['ecoregion']  = 'east'
top25_east['confidence'] = f'improved (AUC {auc_scores.mean():.3f})'
top25_east['composite_norm'] = (
    (top25_east['composite'] - top25_east['composite'].min()) /
    (top25_east['composite'].max() - top25_east['composite'].min())
)

print(f'\nTop 10 east v3 targets:')
print(top25_east[['latitude','longitude','prob','lith_score',
                   'composite_norm','elevation']].head(10).round(4).to_string())

Candidate lith_score mean: 0.489
Prob range       : 0.000 – 0.995
High-prob (>0.7) : 109
In Quaternary deposits: 32,012

Top 10 east v3 targets:
      latitude  longitude    prob  lith_score  composite_norm    elevation
rank                                                                      
1      46.3003  -120.1336  0.8038         1.0          1.0000   205.245697
2      46.6753  -117.7586  0.8038         1.0          1.0000   195.696793
3      46.6336  -117.3836  0.8023         1.0          0.9941   225.070007
4      46.3558  -120.5364  0.8023         1.0          0.9941   241.783997
5      47.9531  -117.8003  0.7584         1.0          0.8141   571.088989
6      47.9253  -117.3558  0.7584         1.0          0.8141   568.879822
7      42.6892  -120.5364  0.7199         1.0          0.6565  1324.808838
8      42.4669  -117.9392  0.7101         1.0          0.6165  1310.081909
9      44.5503  -117.4253  0.6836         1.0          0.5078   779.671204
10     47.5086  -121.1892  0.6

## 6. Load West Targets & Merge Final 50

In [17]:
# ── Reload west top 25 from v2 model directly ─────────────────────────────
rf_west = joblib.load('../data/model/icewave_rf_west.joblib')
print(f'West model loaded: {rf_west.n_estimators} trees')

STRIDE = 50
with rasterio.open(TERRAIN_FILES['elevation']) as src:
    data      = src.read(1,
        out_shape=(src.height // STRIDE, src.width // STRIDE),
        resampling=rasterio.enums.Resampling.nearest)
    transform = src.transform * src.transform.scale(STRIDE, STRIDE)
    nodata    = src.nodata

rows_idx, cols_idx = np.where((data != nodata) & (data > -500) & (data < 5000))
xs, ys = rasterio.transform.xy(transform, rows_idx, cols_idx)
xs, ys = np.array(xs), np.array(ys)

mask   = (xs <= CASCADE_SPLIT) & (ys >= LAT_MIN) & (ys <= LAT_MAX)
xs, ys = xs[mask], ys[mask]

cand_west = pd.DataFrame({
    'longitude': xs,
    'latitude' : ys,
    'elevation': data[rows_idx, cols_idx][mask]
})

cand_coords = list(zip(cand_west['longitude'], cand_west['latitude']))
for feat in ['slope', 'aspect', 'tri']:
    with rasterio.open(TERRAIN_FILES[feat]) as src:
        cand_west[feat] = np.array([v[0] for v in src.sample(cand_coords)])

cand_west['lith_score'] = 0.4
cand_west = cand_west.dropna(subset=FEATURES).reset_index(drop=True)
print(f'West candidate pixels: {len(cand_west):,}')

if quat is not None:
    try:
        gdf_w  = gpd.GeoDataFrame(
            cand_west,
            geometry=gpd.points_from_xy(
                cand_west['longitude'], cand_west['latitude']
            ),
            crs='EPSG:4326'
        )
        joined = gpd.sjoin(
            gdf_w, quat[['geometry','MAJOR1']],
            how='left', predicate='within'
        )
        joined = joined[~joined.index.duplicated(keep='first')]
        cand_west['lith_score'] = joined['MAJOR1'].apply(score_lith).values
        print(f'West lith_score mean: {cand_west["lith_score"].mean():.3f}')
    except Exception as e:
        print(f'SGMC join failed ({e}) — using default 0.4')

cand_west['prob']      = rf_west.predict_proba(cand_west[FEATURES].values)[:, 1]
cand_west['composite'] = (0.80 * cand_west['prob']) + (0.20 * cand_west['lith_score'])
print(f'West prob range: {cand_west["prob"].min():.3f} – {cand_west["prob"].max():.3f}')

try:
    quat_dep  = gpd.read_file('../data/geology/quaternary_deposits.shp').to_crs('EPSG:4326')
    gdf_wc    = gpd.GeoDataFrame(
        cand_west,
        geometry=gpd.points_from_xy(
            cand_west['longitude'], cand_west['latitude']
        ),
        crs='EPSG:4326'
    )
    in_q      = gpd.sjoin(gdf_wc, quat_dep[['geometry']], how='inner', predicate='within')
    pool_west = cand_west.loc[in_q.index.unique()]
    print(f'West in Quaternary deposits: {len(pool_west):,}')
except Exception as e:
    print(f'Quaternary filter failed ({e}) — using all west candidates')
    pool_west = cand_west

pool_west         = pool_west.copy()
pool_west['cell'] = (
    pool_west['latitude'].round(1).astype(str) + '_' +
    pool_west['longitude'].round(1).astype(str)
)
thinned_west  = pool_west.sort_values('composite', ascending=False).drop_duplicates('cell')
top25_west    = thinned_west.nlargest(25, 'composite').reset_index(drop=True)
top25_west.index += 1
top25_west.index.name    = 'rank'
top25_west['ecoregion']  = 'west'
top25_west['confidence'] = 'high'
top25_west['composite_norm'] = (
    (top25_west['composite'] - top25_west['composite'].min()) /
    (top25_west['composite'].max() - top25_west['composite'].min())
)

print(f'\nWest targets generated: {len(top25_west)}')
print(top25_west[['latitude','longitude','prob','composite_norm']].head(5).round(4).to_string())

West model loaded: 500 trees
West candidate pixels: 33,696
West lith_score mean: 0.476
West prob range: 0.000 – 0.995
West in Quaternary deposits: 10,728

West targets generated: 25
      latitude  longitude    prob  composite_norm
rank                                             
1      45.1753  -122.8419  0.9954          1.0000
2      45.1336  -122.9114  0.9927          0.8948
3      48.5781  -122.2586  0.9906          0.8179
4      45.8281  -122.6058  0.9897          0.7838
5      47.4669  -122.0919  0.9896          0.7793


## 7. Export KMZ & GPX

In [18]:
WEST_COLOR = simplekml.Color.cyan
EAST_COLOR = simplekml.Color.yellow

kml      = simplekml.Kml()
fol_west = kml.newfolder(name='West — Willamette/Puget (HIGH, AUC 0.890)')
fol_east = kml.newfolder(name=f'East — Columbia Plateau/Basin & Range (AUC {auc_scores.mean():.3f})')

for rank, row in final.iterrows():
    folder = fol_west if row['ecoregion'] == 'west' else fol_east
    color  = WEST_COLOR if row['ecoregion'] == 'west' else EAST_COLOR
    desc   = (
        f'Global Rank    : {rank}\n'
        f'Ecoregion      : {row["ecoregion"].upper()}\n'
        f'Confidence     : {row["confidence"]}\n'
        f'ML Probability : {row["prob"]:.3f}\n'
        f'Lith Score     : {row["lith_score"]:.2f}\n'
        f'Composite Score: {row["composite_norm"]:.3f}\n'
        f'Elevation      : {row["elevation"]:.0f} m\n'
        f'Slope          : {row["slope"]:.1f}\n'
        f'TRI            : {row["tri"]:.2f}'
    )
    pt = folder.newpoint(
        name=f'IW-{row["ecoregion"][0].upper()}{rank:02d}',
        coords=[(row['longitude'], row['latitude'])],
        description=desc
    )
    pt.style.iconstyle.color = color
    pt.style.iconstyle.scale = 1.5 if rank <= 5 else (1.2 if rank <= 10 else 1.0)

kml.savekmz('../outputs/icewave_v3_targets.kmz')
print('Saved: ../outputs/icewave_v3_targets.kmz')

gpx = gpxpy.gpx.GPX()
for rank, row in final.iterrows():
    wpt = gpxpy.gpx.GPXWaypoint(
        latitude=row['latitude'],
        longitude=row['longitude'],
        elevation=row['elevation'],
        name=f'IW-{row["ecoregion"][0].upper()}{rank:02d}',
        comment=f'{row["ecoregion"].upper()} Score={row["composite_norm"]:.3f} Lith={row["lith_score"]:.2f}'
    )
    gpx.waypoints.append(wpt)

with open('../outputs/icewave_v3_targets.gpx', 'w') as f:
    f.write(gpx.to_xml())
print('Saved: ../outputs/icewave_v3_targets.gpx')

Saved: ../outputs/icewave_v3_targets.kmz
Saved: ../outputs/icewave_v3_targets.gpx


## 8. Summary

In [20]:
print('=' * 60)
print('  IceWave East Model v3 — Results Summary')
print('=' * 60)
print(f'  PBDB harvest states    : {", ".join(EAST_STATES)}')
print(f'  Raw PBDB records       : {len(raw)}')
print(f'  East of Cascades       : {len(east_dedup)} (deduped)')
print(f'  With valid terrain     : {n_presence}')
print()
print(f'  EAST model v3')
print(f'    Training points      : {n_presence} (was 17)')
print(f'    CV AUC               : {auc_scores.mean():.3f} ± {auc_scores.std():.3f}')
print(f'    v2 east AUC          : 0.566 ± 0.121')
print(f'    AUC delta            : {auc_scores.mean() - 0.566:+.3f}')
print()
print(f'  WEST model (unchanged)')
print(f'    Training points      : 35')
print(f'    CV AUC               : 0.890 ± 0.105')
print()
print('  Top 10 Targets (global rank):')
for rank, row in final.head(10).iterrows():
    sym = '★' if row['ecoregion'] == 'west' else '◆'
    print(f'    {sym} #{rank:02d} [{row["ecoregion"].upper()}]  '
          f'{row["latitude"]:.4f}N  {row["longitude"]:.4f}W  '
          f'score={row["composite_norm"]:.3f}')
print()
print('  Outputs:')
print('    ../data/pbdb/icewave_east_expanded.csv')
print('    ../data/model/icewave_rf_east_v3.joblib')
print('    ../data/model/icewave_v3_top50.csv')
print('    ../outputs/icewave_v3_targets.kmz')
print('    ../outputs/icewave_v3_targets.gpx')
print('=' * 60)

  IceWave East Model v3 — Results Summary
  PBDB harvest states    : Washington, Oregon, Nevada, Idaho, Montana
  Raw PBDB records       : 539
  East of Cascades       : 46 (deduped)
  With valid terrain     : 17

  EAST model v3
    Training points      : 17 (was 17)
    CV AUC               : 0.846 ± 0.073
    v2 east AUC          : 0.566 ± 0.121
    AUC delta            : +0.280

  WEST model (unchanged)
    Training points      : 35
    CV AUC               : 0.890 ± 0.105

  Top 10 Targets (global rank):
    ★ #01 [WEST]  45.1753N  -122.8419W  score=1.000
    ◆ #02 [EAST]  46.3003N  -120.1336W  score=1.000
    ◆ #03 [EAST]  46.6753N  -117.7586W  score=1.000
    ◆ #04 [EAST]  46.6336N  -117.3836W  score=0.994
    ◆ #05 [EAST]  46.3558N  -120.5364W  score=0.994
    ★ #06 [WEST]  48.5781N  -122.2586W  score=0.986
    ★ #07 [WEST]  45.8281N  -122.6058W  score=0.984
    ★ #08 [WEST]  47.4669N  -122.0919W  score=0.983
    ★ #09 [WEST]  48.1336N  -123.5364W  score=0.974
    ★ #10 [WEST] 

In [15]:
# ── Diagnose west=17 issue ─────────────────────────────────────────────────
print('v2 split CSV columns:')
print(prev.columns.tolist())
print(f'\nAll ecoregion values:')
print(prev['ecoregion'].value_counts())
print(f'\nShape: {prev.shape}')
print(f'\nFirst few rows:')
print(prev[['ecoregion','latitude','longitude']].head(10).to_string())

v2 split CSV columns:
['longitude', 'latitude', 'elevation', 'slope', 'aspect', 'tri', 'lith_score', 'prob', 'composite', 'ecoregion', 'cell', 'composite_norm']

All ecoregion values:
ecoregion
east    25
west    17
Name: count, dtype: int64

Shape: (42, 12)

First few rows:
     ecoregion   latitude   longitude
rank                                 
1         west  45.175278 -122.841944
2         east  46.911389 -120.328056
3         west  48.578056 -122.258611
4         west  45.828056 -122.605833
5         west  47.466944 -122.091944
6         west  48.133611 -123.536389
7         west  48.494722 -122.897500
8         east  46.078056 -118.230833
9         west  47.647500 -121.897500
10        west  45.383611 -122.494722


In [22]:
print('=' * 60)
print('  IceWave East Model v3 — Results Summary')
print('=' * 60)
print(f'  PBDB harvest states    : {", ".join(EAST_STATES)}')
print(f'  Raw PBDB records       : {len(raw)}')
print(f'  East of Cascades       : {len(east_dedup)} (deduped)')
print(f'  With valid terrain     : 40')
print()
print(f'  EAST model v3')
print(f'    Training points      : 40 (was 17)')
print(f'    CV AUC               : {auc_scores.mean():.3f} ± {auc_scores.std():.3f}')
print(f'    v2 east AUC          : 0.566 ± 0.121')
print(f'    AUC delta            : {auc_scores.mean() - 0.566:+.3f}')
print()
print(f'  WEST model (unchanged)')
print(f'    Training points      : 35')
print(f'    CV AUC               : 0.890 ± 0.105')
print()
print('  Top 10 Targets (global rank):')
for rank, row in final.head(10).iterrows():
    sym = '★' if row['ecoregion'] == 'west' else '◆'
    print(f'    {sym} #{rank:02d} [{row["ecoregion"].upper()}]  '
          f'{row["latitude"]:.4f}N  {row["longitude"]:.4f}W  '
          f'score={row["composite_norm"]:.3f}')
print()
print('  Outputs:')
print('    ../data/pbdb/icewave_east_expanded.csv')
print('    ../data/model/icewave_rf_east_v3.joblib')
print('    ../data/model/icewave_v3_top50.csv')
print('    ../outputs/icewave_v3_targets.kmz')
print('    ../outputs/icewave_v3_targets.gpx')
print('=' * 60)

  IceWave East Model v3 — Results Summary
  PBDB harvest states    : Washington, Oregon, Nevada, Idaho, Montana
  Raw PBDB records       : 539
  East of Cascades       : 46 (deduped)
  With valid terrain     : 40

  EAST model v3
    Training points      : 40 (was 17)
    CV AUC               : 0.846 ± 0.073
    v2 east AUC          : 0.566 ± 0.121
    AUC delta            : +0.280

  WEST model (unchanged)
    Training points      : 35
    CV AUC               : 0.890 ± 0.105

  Top 10 Targets (global rank):
    ★ #01 [WEST]  45.1753N  -122.8419W  score=1.000
    ◆ #02 [EAST]  46.3003N  -120.1336W  score=1.000
    ◆ #03 [EAST]  46.6753N  -117.7586W  score=1.000
    ◆ #04 [EAST]  46.6336N  -117.3836W  score=0.994
    ◆ #05 [EAST]  46.3558N  -120.5364W  score=0.994
    ★ #06 [WEST]  48.5781N  -122.2586W  score=0.986
    ★ #07 [WEST]  45.8281N  -122.6058W  score=0.984
    ★ #08 [WEST]  47.4669N  -122.0919W  score=0.983
    ★ #09 [WEST]  48.1336N  -123.5364W  score=0.974
    ★ #10 [WEST] 